# 24시간 뉴스 수집

보유 종목의 검색어로 구글 뉴스 RSS를 **24시간 검색해 기사를 쌓는** 노트북이다. 묶기·본문·프롬프트 입력 같은 처리는 `news_process.ipynb`가 이 노트북이 쌓은 파일을 읽어서 한다. 정한 것과 근거는 `news_search.md`(1장 한눈에, 2장 수집, 3장 처리, 4장 노트북과 파일), 검색 방식 시험은 `../news_search_tests/search_tests.ipynb`.

**흐름**

```
검색어 표(종목, 검색어, 종류)
  → 마감(1차 5:20, 2차 6:30)에 맞춘 회차마다 검색(검색어마다 3건: 야후 + 5곳 묶음 2)
  → 원본 저장 → 검색 범위보다 오래된 기사·다른 매체 버리기 → 1단계 거르기(제목·매체 주소)
  → 기사 쌓기(같은 매체·같은 제목은 하나로, 검색어마다 바로 저장)
  → 1·2차 마감 회차가 끝나면 마감 표시 파일(cutoff_1차.json·cutoff_2차.json) → news_process.ipynb가 보고 처리
```

**절**

| 절 | 내용 |
|---|---|
| 1 | 설정: 매체 묶음, 야후 제외 문구, 요청 간격·막힘·연결 오류, 검색어 등급, 밤·낮 간격(낮은 8시 30분부터 2시간), 마감, 스위치 |
| 2~7 | 함수: 작은 도구, 검색 문자열, 요청, 응답 읽기·1단계 거르기, 등급과 차례, 한 회차·마감에 맞춘 일정 |
| 8 | 검색어 표와 등급 |
| 9~10 | 한 회차 돌리기(시험), 쌓인 결과 보기 |
| 11 | **24시간 수집 돌리기**: `RUN_COLLECT = True`로 멈출 때까지 |

**쌓이는 파일**(`source/nightly/<보고 날짜>/`, `news_process.ipynb`가 읽는다. 보고 날짜는 그 기사를 보여 주는 아침(1차 6시·2차 7시)의 날짜이고 경계는 2차 마감 6시 30분)

| 파일 | 내용 |
|---|---|
| `raw/<받은 시각>_<해시>.xml.gz` | 응답 원본 |
| `fetch_log.csv` | 요청 기록: 회차, 이름(밤/낮/1차 마감/2차 마감), 검색어, 등급, 묶음(yahoo/A/B), 검색 범위, 받은 시각, 상태, 받은 수, 범위 안 수, 100건 한도 여부, 검색 문자열 |
| `hits.csv` | 어떤 검색어가 어떤 기사를 찾았나(기사 키, 검색어, 회차, 묶음) |
| `items.csv` | 기사(중복 없이): 키, 제목, 매체, 출처 주소, 발행 시각, 구글 링크, 처음 본 시각, 거른 사유(`drop`, 비어 있으면 통과) |
| `cutoff_1차.json`·`cutoff_2차.json` | 마감 표시: 이름, 보고 날짜, 마감 시각, 끝난 시각, 요청 수, 기사 수. 처리 노트북이 이 파일이 생기면 처리한다 |

**한눈에**

| | |
|---|---|
| 외부 요청 | 구글 뉴스 RSS(무료, `RUN`·`RUN_COLLECT = True`일 때만. 시험 종목 18개 기준 하루 약 11,000건) |
| 아직 임시값 | 검색어 등급 기준·간격, 밤·낮 간격(`news_search.md` 결정할 것 1) |
| 지금 한계 | 검색어 표는 3차 시험 목록(시험 종목 18개)을 임시로 쓴다(검색어 표 완성 뒤 `graph_db/search_terms`의 전체 표로 바꾼다) |

## 1. 설정

- **매체**: 야후는 따로 1건, 나머지 10곳은 5곳씩 두 묶음. 묶음이 길면 뒤쪽 매체가 잘린다(10곳 묶음은 8번째까지만 읽힘).
- **야후 제외 문구**: 시세·차트 페이지가 100건 한도를 먹지 않게 제목 문구로 뺀다.
- **요청 간격**: 6초(2.5초는 약 700건째에 막힘). 막히면 20분 쉬고 다시, 6번 넘게 막히면 그 회차를 멈추고 알린다.
- **검색어 등급**: 기사가 많을수록 자주, 짧은 검색 범위로. 검색 범위(`when:`)는 검색 간격보다 1시간 길게 잡아 앞 회차와 겹치게 한다. 기준값은 임시다.

In [1]:
# ── 불러오기 ──
import datetime as dt
import gzip
import hashlib
import json
import random
import re
import time
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests

pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)
RUN = False          # True여야 구글 뉴스 RSS 요청을 보낸다

In [2]:
# ── 검색 설정 ──
ROOT = Path.cwd()                               # news_search 폴더(노트북을 이 폴더에서 연다)
PROJ = ROOT.parent                              # 프로젝트 루트(.env가 있는 곳)
NIGHTLY_DIR = ROOT / 'source' / 'nightly'       # 보고 날짜마다 폴더 하나(원본 응답, 요청 기록, 쌓인 기사)
KST = ZoneInfo('Asia/Seoul')
ET = ZoneInfo('America/New_York')

# 검색 매체: 야후는 따로 1건, 나머지 10곳은 5곳씩 두 묶음(기사 많은 매체를 두 묶음에 나눠 섞음).
# 묶음이 길면 뒤쪽 매체가 잘린다(10곳 묶음은 8번째까지만 읽힘). 5곳이면 3단어 검색어여도 여유가 있다.
YAHOO = 'finance.yahoo.com'
GROUP_A = ['fool.com', 'cnbc.com', 'theguardian.com', 'businessinsider.com', 'foxbusiness.com']
GROUP_B = ['fortune.com', 'cbsnews.com', 'nypost.com', 'usatoday.com', 'cnn.com']
SITES = [YAHOO] + GROUP_A + GROUP_B
SITE_NAMES = {'finance.yahoo.com': 'Yahoo Finance', 'fool.com': 'Motley Fool', 'cnbc.com': 'CNBC', 'theguardian.com': 'Guardian',
              'businessinsider.com': 'Business Insider', 'foxbusiness.com': 'Fox Business', 'fortune.com': 'Fortune',
              'cbsnews.com': 'CBS News', 'nypost.com': 'NY Post', 'usatoday.com': 'USA Today', 'cnn.com': 'CNN'}

# 야후 시세·차트 페이지는 제목 문구로 검색에서 뺀다(큰 종목은 이것들이 100건 한도를 먹는다).
# 주소 조건(-inurl:quote)은 구글 뉴스에서 먹히지 않았다.
YAHOO_EXCLUDE = '-"Stock Price, News, Quote" -"Latest Stock News" -"META_TITLE_QUOTE"'

# 요청 간격과 막혔을 때. 6초 간격은 막히지 않았고, 2.5초는 약 700건째에 503으로 2시간 가까이 막혔다.
RSS_SLEEP = 6               # 요청 사이 간격(초, ±0.5초 무작위)
BLOCK_WAIT = 1200           # 429·503이면 이만큼(초) 쉬고 다시
MAX_BLOCK_WAITS = 6         # 쉬는 최대 횟수(넘으면 그 회차를 멈추고 알린다)
CAP = 100                   # 구글 뉴스 RSS 한 요청의 최대 기사 수
NET_WAIT = 60               # 연결 오류(네트워크가 바뀌거나 끊김)면 이만큼(초) 쉬고 다시
MAX_NET_RETRIES = 5         # 연결 오류로 다시 시도하는 최대 횟수(넘으면 그 회차를 멈춘다)

# 검색어 등급: 기사가 많을수록 자주, 짧은 검색 범위로 검색한다. 검색 범위(when:)는 검색 간격보다 1시간 길게 잡아 앞 회차와 겹치게 한다
# (검색에 뜨는 지연, 실행이 조금 늦는 경우에도 빠지지 않게).
# 등급을 나누는 하루 기사 수 기준은 임시값이다. 평일 미국 낮 실시간 시험으로 정한다(news_search.md 결정할 것 1).
TIERS = {
    '1h': dict(every_h=1, window_h=2),
    '3h': dict(every_h=3, window_h=4),
    '6h': dict(every_h=6, window_h=7),
    '12h': dict(every_h=12, window_h=13),
}
TIER_BY_DAILY = [(100, '1h'), (30, '3h'), (10, '6h'), (0, '12h')]   # (하루 기사 수 이상, 등급), 위에서부터 처음 맞는 것

# 시간대별 등급('시간대별'): 등급을 정하기 전 시험용. 모든 검색어를 밤에는 자주, 낮에는 드물게 검색한다.
# 밤(한국 16시~다음 날 8시 30분, 미국 새벽 2시~오후 6시 30분): 미국 장중과 장 마감 직후. 낮(한국 8시 30분~16시): 기사가 적다.
NIGHT_START_KST = 16        # 밤 시작(한국 시간)
DAY_START_KST = (8, 30)     # 낮 시작(한국 시, 분): 이때부터 검색 간격을 늘린다. 2차 마감(6:30) 뒤 이때까지는 1시간 간격(장 마감 뒤 실적 발표가 몰리는 때)
DAY_NIGHT = {
    '밤': dict(every_h=1, window_h=2),
    '낮': dict(every_h=2, window_h=3),
}
# 마감에 맞춘 일정: 결과는 아침 6시(1차)·7시(2차)에 보여 준다. 마감 뒤 처리(묶기·본문·프롬프트)에 약 30~40분이 걸리므로
# 수집은 그보다 앞서 끝나야 한다. 회차 시작 시각을 마감에서 거꾸로 잡아 마지막 회차가 마감에 딱 끝나게 한다.
CUTOFFS = [('1차', 5, 20), ('2차', 6, 30)]    # (이름, 시, 분): 한국 시간. 1차는 장중 소식, 2차는 장 마감 시황·마감 직후 발표(겨울 기준)
REPORT_BOUNDARY = (6, 30)   # 보고 날짜 경계(2차 마감). 이 시각 전까지 모은 기사가 그날 아침 보고에 들어간다
DAY_EVERY_H = 2             # 낮(8시 30분~16시) 회차 간격
NIGHT_EVERY_H = 1           # 밤 회차 간격
WINDOW_MARGIN_H = 1         # 검색 범위 = 앞 회차 시작부터 지금까지 + 이만큼(빠짐없이 겹치게)
SEC_PER_REQUEST = RSS_SLEEP + 0.4   # 회차 길이를 어림할 때 요청 1건에 걸리는 시간(초). 회차를 돌고 나면 실제 값으로 바꿔 쓴다
assert ROOT.name == 'news_search', 'news_search 폴더에서 실행해야 한다.'

## 2. 작은 도구

보고 날짜(2차 마감 6시 30분 경계), 밤·낮 구분과 시간대별 검색 간격, `.env` 읽기(값은 화면·파일에 쓰지 않음), 텔레그램 알림(실패해도 멈추지 않고 오류에서 토큰을 가림), CSV 이어 쓰기.

In [3]:
text_hash = lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16]
now_utc = lambda: dt.datetime.now(dt.timezone.utc)


def report_id(when=None):
    """보고 날짜: 이 기사들을 보여 주는 아침(1차 6시·2차 7시)의 날짜. 경계는 2차 마감(6시 30분).
    예: 11-15 23:00 → 20261116, 11-16 06:29(2차 회차) → 20261116, 11-16 07:30(낮) → 20261117."""
    t = (when or now_utc()).astimezone(KST)
    if (t.hour, t.minute) >= REPORT_BOUNDARY:
        t += dt.timedelta(days=1)
    return t.strftime('%Y%m%d')


def day_or_night(when=None):
    """지금이 밤('밤')인지 낮('낮')인지(한국 시간 기준). 낮은 DAY_START_KST부터 NIGHT_START_KST 전까지."""
    t = (when or now_utc()).astimezone(KST)
    return '낮' if DAY_START_KST <= (t.hour, t.minute) < (NIGHT_START_KST, 0) else '밤'


def tier_params(tier, when=None):
    """등급 → 검색 간격·범위. '시간대별'이면 지금 시간대(밤/낮)의 값."""
    return DAY_NIGHT[day_or_night(when)] if tier == '시간대별' else TIERS[tier]


def night_dir(nid):
    d = NIGHTLY_DIR / nid
    (d / 'raw').mkdir(parents=True, exist_ok=True)
    return d


def read_env():
    """프로젝트 루트 .env를 읽는다(값은 화면·파일에 쓰지 않는다)."""
    env = {}
    p = PROJ / '.env'
    if p.exists():
        for line in p.read_text(encoding='utf-8-sig').splitlines():
            k, _, v = line.partition('=')
            env[k.strip()] = v.strip().strip('"\'')
    return env


def notify(text):
    """텔레그램 알림(.env의 ALERT_TELEGRAM_TOKEN·ALERT_TELEGRAM_CHAT_ID). 실패해도 멈추지 않고, 오류에서 토큰을 가린다."""
    env = read_env()
    token, chat = env.get('ALERT_TELEGRAM_TOKEN'), env.get('ALERT_TELEGRAM_CHAT_ID')
    print('알림:', text)
    if token and chat:
        try:
            requests.post(f'https://api.telegram.org/bot{token}/sendMessage', data={'chat_id': chat, 'text': text}, timeout=20)
        except requests.RequestException as e:
            print('  텔레그램 알림 실패:', str(e).replace(token, '***')[:200])


def append_csv(df, path):
    """표를 CSV 끝에 붙인다(처음이면 머리줄과 함께 만든다)."""
    if len(df):
        df.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')

## 3. 검색 문자열

검색어 하나 → 3건: 야후(시세 페이지 제목 문구를 뺀다) + 묶음 A + 묶음 B. `when:`은 OR 묶음 앞에 둬야 지켜진다(뒤에 쓰면 무시되어 옛 기사가 온다). 검색어는 검색어 표에 적힌 그대로 쓴다(구문은 큰따옴표).

In [4]:
def or_group(sites):
    return '(' + ' OR '.join(f'site:{s}' for s in sites) + ')'


def build_queries(term, window_h):
    """검색어 하나 → 검색 문자열 3개(야후, 묶음 A, 묶음 B).

    `when:`은 OR 묶음 앞에 써야 지켜진다(뒤에 쓰면 무시되어 옛 기사가 온다).
    term은 검색어 표에 적힌 그대로 쓴다(구문은 큰따옴표로 묶여 있다: "SK hynix").
    """
    w = f'when:{window_h}h'
    return [
        dict(group='yahoo', query=f'{term} {w} site:{YAHOO} {YAHOO_EXCLUDE}'),
        dict(group='A', query=f'{term} {w} {or_group(GROUP_A)}'),
        dict(group='B', query=f'{term} {w} {or_group(GROUP_B)}'),
    ]

In [5]:
# ── 검색 문자열 예 ──
for q in build_queries('"SK hynix"', TIERS['1h']['window_h']):
    print(f"{q['group']:6} {q['query']}")

yahoo  "SK hynix" when:2h site:finance.yahoo.com -"Stock Price, News, Quote" -"Latest Stock News" -"META_TITLE_QUOTE"
A      "SK hynix" when:2h (site:fool.com OR site:cnbc.com OR site:theguardian.com OR site:businessinsider.com OR site:foxbusiness.com)
B      "SK hynix" when:2h (site:fortune.com OR site:cbsnews.com OR site:nypost.com OR site:usatoday.com OR site:cnn.com)


## 4. 요청 보내기

외부 요청은 여기서만 보낸다. 요청 간격을 지키고, 429·503이면 쉬었다가 다시 하고, 쉬어도 계속 막히면 상태 코드를 그대로 돌려줘 회차를 멈추게 한다.

In [6]:
class GoogleNewsClient:
    """구글 뉴스 RSS 요청.

    - 요청 간격을 지킨다(RSS_SLEEP).
    - 429·503(막힘)이면 BLOCK_WAIT 쉬고 다시, 연달아 MAX_BLOCK_WAITS번 넘게 막히면 상태 코드를 그대로 돌려줘 회차를 멈추게 한다.
    - 연결 오류(네트워크가 바뀌거나 끊김)면 NET_WAIT 쉬고 다시, MAX_NET_RETRIES번 넘게 실패하면 '연결 끊김'을 돌려준다.
      다시 연결되면 끊긴 시간을 텔레그램으로 알린다(끊긴 동안에는 알림을 보낼 수 없어서).
    """

    HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0 Safari/537.36',
               'Accept': 'application/rss+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-US,en;q=0.9'}

    def __init__(self, sleep=RSS_SLEEP):
        self.ses = requests.Session()
        self.ses.headers.update(self.HEADERS)
        self.sleep = sleep
        self.last = 0.0          # 마지막 요청 시각(time.time)
        self.block_waits = 0     # 연달아 막혀서 쉰 횟수(성공하면 0으로)
        self.offline_since = None  # 연결이 끊기기 시작한 시각(다시 연결되면 알리고 지운다)

    def get(self, query, deadline=None):
        """검색 문자열 하나를 요청한다. 반환: (상태 코드 또는 오류 이름, 응답 바이트 또는 None).
        deadline(한국 시간)이 있으면 쉬는 시간이 그 시각을 넘길 때 쉬지 않고 바로 돌려준다(마감을 지키려고)."""
        past = lambda sec: deadline is not None and dt.datetime.now(KST) + dt.timedelta(seconds=sec) > deadline
        url = 'https://news.google.com/rss/search?q=' + quote_plus(query) + '&hl=en-US&gl=US&ceid=US:en'
        net_tries = 0
        while True:
            wait = self.last + self.sleep + random.uniform(-0.5, 0.5) - time.time()
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()
            try:
                r = self.ses.get(url, timeout=30)
                status, body = r.status_code, r.content
            except requests.RequestException:
                # 연결 오류: 1분 쉬고 다시(최대 MAX_NET_RETRIES번)
                self.offline_since = self.offline_since or dt.datetime.now(KST)
                if net_tries < MAX_NET_RETRIES and not past(NET_WAIT):
                    net_tries += 1
                    print(f'  {dt.datetime.now(KST):%H:%M} 연결 오류 → {NET_WAIT}초 쉬고 다시({net_tries}/{MAX_NET_RETRIES})')
                    time.sleep(NET_WAIT)
                    continue
                return '연결 끊김', None
            if self.offline_since:              # 다시 연결됨: 끊긴 시간을 알린다
                notify(f'[news_search] 네트워크가 {self.offline_since:%H:%M}부터 {dt.datetime.now(KST):%H:%M}까지 끊겼다가 다시 연결됐다.')
                self.offline_since = None
            if status in (429, 503) and self.block_waits < MAX_BLOCK_WAITS and not past(BLOCK_WAIT):
                self.block_waits += 1
                print(f'  {dt.datetime.now(KST):%H:%M} {status}로 막힘 → {BLOCK_WAIT // 60}분 쉬고 다시({self.block_waits}/{MAX_BLOCK_WAITS})')
                time.sleep(BLOCK_WAIT)
                continue
            if status == 200:
                self.block_waits = 0             # 막힘이 풀렸으면 횟수를 되돌린다(하루 넘게 돌 때 쌓이지 않게)
            return status, (body if status == 200 else None)

## 5. 응답 읽기와 1단계 거르기

- 응답에는 원문 주소가 없고 구글 뉴스 링크만 있다(원문 주소는 본문 받기에서 복원한다). 출처 주소는 우리 매체 이름으로 맞춘다(야후 지역판 → finance.yahoo.com).
- **1단계 거르기**는 제목과 매체 주소로 한다: 시세·자료 페이지, 실적 발표 대본, 틀 이름만 있는 빈 제목, 팟캐스트·사진, 쇼핑, 행사 홍보, 인사 동정, 팬 사이트·보도자료·지역판 복사본. 원문 주소(2단계)·본문(3단계) 규칙은 본문 받기에서 쓴다(`news_search.md` 3장).

In [7]:
def base_outlet(domain):
    """출처 주소 → 우리 매체 이름(지역판 ca.finance.yahoo.com → finance.yahoo.com). 11곳이 아니면 주소 그대로."""
    d = (domain or '').lower()
    return next((s for s in SITES if d == s or d.endswith('.' + s)), d)


def clean_title(title):
    """구글 뉴스 제목 끝의 ' - 매체 이름'을 뗀다."""
    return re.sub(r'\s+-\s+[^-]+$', '', title or '').strip()


def parse_items(body):
    """응답 하나 → 기사 목록. 원문 주소는 없고 구글 뉴스 링크만 있다(원문 주소는 본문 받기에서 복원)."""
    out = []
    for e in feedparser.parse(body).entries:
        try:
            pub = parsedate_to_datetime(e.get('published')).astimezone(dt.timezone.utc)
        except Exception:
            pub = None
        src = e.get('source') or {}
        domain = re.sub(r'^https?://(www\.)?', '', src.get('href', '')).strip('/').lower()
        out.append(dict(title=clean_title(e.get('title', '')), source=src.get('title', ''), domain=domain,
                        outlet=base_outlet(domain), published_utc=pub.isoformat() if pub else '', link=e.get('link', '')))
    return out

# 검색 결과에는 제목과 매체 주소만 있다. 원문 주소(2단계)·본문(3단계) 규칙은 본문 받기에서 쓴다(news_search.md 3장).

TITLE_RULES = [
    # 시험에서 본 꼴: 시세·차트·자료 페이지(야후 제외 문구로 대부분 빠지지만 남는 것)
    ('시세·자료 페이지', r'stock price, news, quote|quote & history|quote and history|price history|seasonal returns chart'
                       r'|interactive stock chart|charts, data (and|&) news|live price, news, chart|stock forum|analyst insights'
                       r'|price targets & recommendations|historical prices|options chain|key statistics|stock news today'
                       r'|latest stock news|stock news & headlines|stock forecasts$|\bstock performance$|^yahoo finance$'),
    ('실적 발표 대본', r'earnings call transcript|call transcript$|prepared remarks'),
    ('빈 제목', r'(?-i:^[A-Z0-9]+(?:_[A-Z0-9]+)+$)'),        # META_TITLE_QUOTE 같은 틀 이름(대소문자를 가린다)
    # graph_db/news_filter.ipynb의 제목 규칙
    ('영상·팟캐스트·사진', r'^podcast:|^in pictures:| in photos$|^in the know \d+'),
    ('생활·쇼핑', r'best .*gifts?\b|gift sets'),
    ('매체 홍보·행사 안내', r'^register:|final call for entries|webinar|free ebook|^get the .* newsletter'),
    ('인사 동정 모음', r'^people moves|^people on the move|^jobs roundup'),
]
HOST_RULES = [   # 매체 주소(하위 도메인 포함): 언론사 시험의 매체별 제외 구역 중 주소로 알 수 있는 것
    ('스포츠 팬 사이트', r'^ftw\.usatoday\.com$|wire\.usatoday\.com$'),
    ('여행·생활 목록', r'^10best\.usatoday\.com$'),
    ('보도자료', r'^markets\.businessinsider\.com$'),
    ('방송 대본·행사', r'^transcripts\.cnn\.com$|^cnnicevents\.'),
    ('지역판 복사본', r'^africa\.businessinsider\.com$'),
]
TITLE_RX = [(r, re.compile(x, re.I)) for r, x in TITLE_RULES]
HOST_RX = [(r, re.compile(x, re.I)) for r, x in HOST_RULES]


def drop_reason(title, domain=''):
    """기사가 아니라고 볼 사유(없으면 빈 글자)."""
    return (next((r for r, rx in HOST_RX if rx.search(domain or '')), '')
            or next((r for r, rx in TITLE_RX if rx.search(title or '')), ''))

## 6. 검색어 등급과 차례

- **등급**: 하루 기사 수로 매긴다(100건 이상 1시간마다, 30~100 3시간, 10~30 6시간, 그 아래 12시간). 구글은 한도(100건)에 걸리면 최신 100건이 아니라 표본을 주므로, 기사가 많은 검색어일수록 짧은 검색 범위로 자주 검색해야 빠지지 않는다.
- **차례**: 검색어마다 마지막으로 3건(야후·A·B)을 다 받은 시각에서 등급 간격이 지나면 다시 검색한다. 막혀서 중간에 멈춘 검색어는 다음 회차에 다시 한다.

In [8]:
def tier_of(daily_articles):
    """하루 기사 수 → 등급(1h/3h/6h/12h)."""
    return next(t for lo, t in TIER_BY_DAILY if daily_articles >= lo)


def requests_per_day(terms):
    """검색어 표(term, tier) → 하루(24시간) 예상 요청 수: 검색어마다 3건 × 하루 검색 횟수.
    '시간대별'은 밤 시간 ÷ 밤 간격 + 낮 시간 ÷ 낮 간격."""
    night_h = (DAY_START_KST[0] + DAY_START_KST[1] / 60 + 24 - NIGHT_START_KST) % 24
    def rounds(t):
        if t == '시간대별':
            return night_h / DAY_NIGHT['밤']['every_h'] + (24 - night_h) / DAY_NIGHT['낮']['every_h']
        return 24 / TIERS[t]['every_h']
    return int(3 * terms.drop_duplicates('term')['tier'].map(rounds).sum())


def due_terms(terms, fetch_log, now=None):
    """지금 검색할 차례인 검색어. 마지막으로 3건을 모두 받은 시각에서 등급 간격이 지났거나, 이 보고 날짜 폴더에서 아직 안 받은 것."""
    now = now or now_utc()
    last = {}                                                   # 검색어 → 마지막으로 3건을 다 받은 시각
    if len(fetch_log):
        ok = fetch_log[fetch_log['status'].astype(str) == '200']
        full = ok.groupby(['term', 'round']).filter(lambda g: g['group'].nunique() == 3)   # 3건(야후·A·B)을 다 받은 회차만
        last = full.groupby('term')['fetched_utc'].max().map(pd.Timestamp).to_dict()
    due = []
    for r in terms.drop_duplicates('term').itertuples():
        every = dt.timedelta(hours=tier_params(r.tier, now)['every_h']) - dt.timedelta(minutes=5)   # 회차 확인 주기 때문에 5분 여유
        if r.term not in last or now - last[r.term] >= every:
            due.append(r.term)
    return due

## 7. 한 회차·마감에 맞춘 일정

- `run_round`: 차례가 된 검색어를 검색해 이 보고 날짜 폴더에 쌓는다. 검색 범위보다 오래된 기사(야후의 몇 년 전 시세 페이지 등)와 11곳이 아닌 매체는 버린다. 같은 매체의 같은 제목은 기사 하나로 본다(야후 지역판 주소 중복).
- `run_scheduled`: **마감에 맞춘 일정**으로 멈출 때까지 수집한다. 결과는 1차 6시(장중 소식), 2차 7시(장 마감 시황·마감 직후 발표)에 보여 주므로 수집은 1차 5시 20분, 2차 6시 30분에 끝나야 한다.
  - 회차 시작 시각을 마감에서 거꾸로 잡는다(`plan_rounds`). 회차 길이 = 검색어 수 × 3건 × 직전 회차의 실제 요청 간격. 예(회차 55분): 밤 1:25, 2:25, 3:25 → 1차 마감 회차 4:25~5:20 → 2차 마감 회차 5:35~6:30 → 장 마감 직후 7:30, 8:30(1시간 간격) → 낮 10:30, 12:30, 14:30(2시간 간격) → 다시 밤 16:25부터. 회차가 간격보다 길면(검색어가 많을 때) 쉬지 않고 이어서 돈다.
  - 검색 범위는 검색어마다 "마지막으로 받은 때부터 지금까지 + 1시간"(빠짐없이 겹치게). 회차 안에서는 조용한 검색어부터, 기사가 많은 검색어는 끝에(마감 직전까지의 기사를 담게).
  - **마감을 지킨다**: 회차마다 끝나야 할 시각이 있다(마감 회차는 1차 5:20·2차 6:30, 나머지는 다음 회차 시작). 늦게 시작하면 남은 시간에 맞춰 기사가 많은 검색어부터 하고, 그 시각이 되면 그 자리에서 멈춘다(못 한 검색어는 다음 회차에서). 막힘·연결 오류로 쉬는 시간이 마감을 넘기면 쉬지 않고 멈춘다. 검색어마다 바로 저장하므로 마감 시각에는 받은 데까지 파일에 있다.
  - **마감 뒤 바로 다음 처리로**: 마감 회차가 끝나면 `on_cutoff`가 표시 파일(`cutoff_1차.json`·`cutoff_2차.json`)을 남기고 알린다. 처리는 `news_process.ipynb`가 이 파일이 생기면 한다(수집과 처리를 나눠, 처리 코드를 고쳐도 수집을 멈추지 않는다).
  - 다음 회차는 방금 한 회차의 예정 시각 뒤에서 고른다. 1차가 늦어져도 2차 마감 회차는 사라지지 않는다.
  - 보고 날짜 경계는 2차 마감(6시 30분): 1차·2차가 같은 보고 날짜 폴더에 들어간다.
  - 마감 회차가 끝나면, 그리고 막히거나 네트워크가 끊겼다가 돌아오면 텔레그램으로 알린다.
- `load_night`: 쌓인 결과(기사, 검색어–기사 연결, 요청 기록)를 읽는다.

In [9]:
def run_round(terms, client, nid=None, limit=None, log=print, todo=None, window_h=None, label='', deadline=None):
    """차례가 된 검색어를 검색하고 결과를 이 보고 날짜 폴더에 쌓는다.

    terms: 검색어 표. 칸: term(검색 문자열에 넣을 검색어), tier, company, kind (한 검색어가 여러 종목에 걸리면 여러 줄)
    todo를 주면(마감에 맞춘 일정) 그 검색어들을, 안 주면 등급에 따라 차례가 된 검색어를 검색한다.
    검색 범위: window_h를 주면 그 값, todo만 주면 검색어마다 "마지막으로 받은 때부터 지금까지 + 여유"(term_window), 둘 다 없으면 등급 값.
    검색어는 조용한 것부터, 기사가 많은 것은 끝에 검색한다(마감 직전까지의 기사를 담게).
    deadline(한국 시간): 이 시각이 되면 그 자리에서 멈춘다. 늦게 시작해 다 못 하면 기사가 많은 검색어부터 골라
    남은 시간에 맞춘다(못 한 검색어는 다음 회차에서, 범위는 마지막으로 받은 때부터라 빠지지 않는다).
    검색어마다 바로 저장하므로 마감 시각에는 받은 데까지 파일에 있다.
    쌓는 파일(source/nightly/<보고 날짜>/):
      raw/<받은 시각>_<해시>.xml.gz  응답 원본(같은 검색 문자열도 회차마다 따로 남긴다)
      fetch_log.csv                  요청 기록: 회차, 검색어, 묶음, 검색 범위, 받은 시각, 상태, 받은 수
      hits.csv                       어떤 검색어가 어떤 기사를 찾았나: 기사 키, 검색어, 회차
      items.csv                      기사(중복 없이): 키, 제목, 매체, 발행 시각, 링크, 처음 본 시각, 거른 사유
    """
    nid = nid or report_id()
    d = night_dir(nid)
    log_path, hits_path, items_path = d / 'fetch_log.csv', d / 'hits.csv', d / 'items.csv'
    fetch_log = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame(columns=['term', 'round', 'group', 'status', 'fetched_utc'])
    per_term_window = todo is not None and window_h is None
    todo = list(todo) if todo is not None else due_terms(terms, fetch_log)
    if limit:
        todo = todo[:limit]
    if deadline is not None:                       # 남은 시간에 다 못 하면 기사가 많은 검색어부터 고른다
        spr = seconds_per_request(fetch_log)
        n_fit = max(0, int((deadline - dt.datetime.now(KST)).total_seconds() // (3 * spr)))
        if n_fit < len(todo):
            busy_first = order_by_volume(todo, fetch_log)[::-1]
            log(f'  남은 시간에 검색어 {n_fit}/{len(todo)}개만 할 수 있어 기사가 많은 것부터 고른다(나머지는 다음 회차)')
            todo = busy_first[:n_fit]
    todo = order_by_volume(todo, fetch_log)
    round_id = now_utc().strftime('%Y%m%dT%H%M%S')
    tier = terms.drop_duplicates('term').set_index('term')['tier']
    known = set(pd.read_csv(items_path, usecols=['key'])['key']) if items_path.exists() else set()
    log(f'[{nid}] 회차 {round_id}{(" " + label) if label else ""}: 검색어 {len(todo)}개 × 3건 = 요청 {3 * len(todo)}건 (예상 {3 * len(todo) * client.sleep / 60:.0f}분)')
    logs_all, n_new = [], 0
    blocked = False
    for i, term in enumerate(todo, 1):
        if deadline is not None and dt.datetime.now(KST) >= deadline:
            log(f'  마감({deadline:%H:%M})이 되어 멈춤: {i - 1}/{len(todo)}개 검색')
            break
        logs, hits, items = [], [], []
        window_h_term = window_h or (term_window(term, fetch_log) if per_term_window else tier_params(tier[term])['window_h'])
        for q in build_queries(term, window_h_term):
            fetched = now_utc()
            status, body = client.get(q['query'], deadline=deadline)
            n_raw = n_kept = 0
            if body is not None:
                # 원본은 받은 그대로 남긴다(같은 검색 문자열도 회차마다 결과가 달라 받은 시각을 이름에 넣는다)
                (d / 'raw' / f'{fetched:%Y%m%dT%H%M%S}_{text_hash(q["query"])}.xml.gz').write_bytes(gzip.compress(body))
                found = parse_items(body)
                n_raw = len(found)
                for it in found:
                    # 검색 범위보다 오래된 기사(야후의 몇 년 전 시세 페이지 등)는 버린다. 범위보다 30분 여유를 둔다.
                    if not it['published_utc'] or fetched - dt.datetime.fromisoformat(it['published_utc']) > dt.timedelta(hours=window_h_term, minutes=30):
                        continue
                    if it['outlet'] not in SITES:          # 우리 매체가 아닌 것이 섞여 오면 버린다
                        continue
                    # 같은 매체의 같은 제목 = 같은 기사(야후 지역판 주소처럼 주소만 다른 중복을 하나로)
                    key = text_hash(it['outlet'] + '|' + it['title'].lower())
                    hits.append(dict(key=key, term=term, round=round_id, group=q['group']))
                    if key not in known:
                        known.add(key)
                        items.append(dict(key=key, **it, first_seen_utc=fetched.isoformat(), drop=drop_reason(it['title'], it['domain'])))
                    n_kept += 1
            logs.append(dict(round=round_id, label=label, term=term, tier=tier[term], group=q['group'], window_h=window_h_term,
                             fetched_utc=fetched.isoformat(), status=status, n=n_raw, n_in_window=n_kept,
                             capped=n_raw >= CAP, query=q['query']))
            if status in (429, 503, '연결 끊김'):          # 쉬었다가 다시 해도 계속 막힘·끊김(또는 마감 때문에 못 기다림)
                blocked = status
                break
        # 검색어마다 바로 저장(마감·중단 때 받은 데까지 남게)
        append_csv(pd.DataFrame(logs), log_path)
        append_csv(pd.DataFrame(hits), hits_path)
        append_csv(pd.DataFrame(items), items_path)
        logs_all += logs
        n_new += len(items)
        if blocked:
            why = '구글 뉴스 RSS가 계속 막혀' if blocked != '연결 끊김' else '네트워크 연결이 계속 안 돼'
            notify(f'[news_search] {nid}: {why} 이번 회차를 멈췄다({i}/{len(todo)}). 다음 회차에 남은 것부터 한다.')
            break
        if i % 20 == 0:
            log(f'  {i}/{len(todo)}')
    L = pd.DataFrame(logs_all)
    if len(L):
        capped = int(L['capped'].sum())
        log(f'  받음 {(L["status"].astype(str) == "200").sum()}/{len(L)}건, 새 기사 {n_new}건'
            + (f', 100건 한도에 걸린 요청 {capped}건(그 검색어는 등급을 올려야 한다)' if capped else ''))
    return L


def load_night(nid):
    """쌓인 결과 읽기: (기사, 검색어–기사 연결, 요청 기록). 거른 기사는 빼지 않고 drop 칸으로 표시돼 있다."""
    d = NIGHTLY_DIR / nid
    cols = {'items.csv': ['key', 'title', 'source', 'domain', 'outlet', 'published_utc', 'link', 'first_seen_utc', 'drop'],
            'hits.csv': ['key', 'term', 'round', 'group'],
            'fetch_log.csv': ['round', 'label', 'term', 'tier', 'group', 'window_h', 'fetched_utc', 'status', 'n', 'n_in_window', 'capped', 'query']}
    # 파일이 없거나 비어 있어도 칸은 갖춘 빈 표를 돌려준다(뒤 단계가 칸 이름으로 읽는다)
    read = lambda f: pd.read_csv(d / f) if (d / f).exists() else pd.DataFrame(columns=cols[f])
    items = read('items.csv')
    items['drop'] = items['drop'].fillna('')
    return items, read('hits.csv'), read('fetch_log.csv')


def summary_text(nid):
    items, hits, flog = load_night(nid)
    return f'요청 {len(flog)}건, 기사 {len(items)}건(거른 것 {(items["drop"] != "").sum()}건)'


def order_by_volume(todo, fetch_log):
    """검색어를 조용한 것부터 기사가 많은 것 순으로(지난 기록의 회차당 평균 기사 수). 기록이 없으면 그대로."""
    if not len(fetch_log) or 'n' not in fetch_log:
        return list(todo)
    vol = fetch_log.groupby('term')['n'].mean()
    return sorted(todo, key=lambda t: (vol.get(t, 0), t))


def term_window(term, fetch_log, now=None):
    """검색어 하나의 검색 범위(시간): 마지막으로 3건을 다 받은 때부터 지금까지 + WINDOW_MARGIN_H(올림). 기록이 없으면 밤 간격 + 여유. 최대 24시간."""
    now = now or now_utc()
    if len(fetch_log):
        ok = fetch_log[(fetch_log['term'] == term) & (fetch_log['status'].astype(str) == '200')]
        full = ok.groupby('round').filter(lambda g: g['group'].nunique() == 3)
        if len(full):
            last = pd.Timestamp(full.groupby('round')['fetched_utc'].min().max())   # 마지막으로 다 받은 회차에서 그 검색어를 처음 요청한 시각
            since = (now - last).total_seconds() / 3600
            return min(24, int(-(-since // 1)) + WINDOW_MARGIN_H)
    return NIGHT_EVERY_H + WINDOW_MARGIN_H


def seconds_per_request(fetch_log):
    """실제 요청 1건에 걸린 시간(초): 마지막 회차의 받은 시각 간격 중앙값. 없으면 SEC_PER_REQUEST."""
    if len(fetch_log) < 10:
        return SEC_PER_REQUEST
    last = fetch_log[fetch_log['round'] == fetch_log['round'].iloc[-1]]
    gaps = pd.to_datetime(last['fetched_utc']).diff().dt.total_seconds().dropna()
    gaps = gaps[gaps < 60]                    # 막혀서 쉰 시간은 뺀다
    return float(gaps.median()) if len(gaps) >= 5 else SEC_PER_REQUEST


def plan_rounds(start, round_min, days=2):
    """start(한국 시간) 이후 days일 동안의 회차: [(시작 시각, 이름, 끝나야 할 시각)]. start보다 이른 회차는 뺀다.
    끝나야 할 시각: 마감 회차는 그 마감(1차 5:20, 2차 6:30), 나머지는 다음 회차 시작(늦은 회차가 뒤 일정을 밀지 않게).

    마감에서 거꾸로 잡는다(회차 길이 round_min분):
      - 1차 마감(5:20)에 끝나도록 밤 회차를 1시간 간격(회차가 더 길면 회차 길이)으로 거슬러 16시까지
      - 2차 마감(6:30)에 끝나는 회차 하나(1차 회차가 끝난 뒤에 시작)
      - 2차 뒤 낮 시작(8:30)까지는 1시간 간격(7:30, 8:30), 그 뒤 낮은 2시간 간격으로 16시 전까지(10:30, 12:30, 14:30)
    """
    L = dt.timedelta(minutes=round_min)
    night_step = max(dt.timedelta(hours=NIGHT_EVERY_H), L)   # 회차가 간격보다 길면(검색어가 많을 때) 쉬지 않고 이어서
    day_step = max(dt.timedelta(hours=DAY_EVERY_H), L)
    plan = []
    day0 = start.astimezone(KST).date()
    for k in range(-1, days + 1):
        d = day0 + dt.timedelta(days=k)
        at = lambda h, m=0, dd=d: dt.datetime(dd.year, dd.month, dd.day, h, m, tzinfo=KST)
        (n1, h1, m1), (n2, h2, m2) = CUTOFFS
        c1, c2 = at(h1, m1), at(h2, m2)
        night_start = at(NIGHT_START_KST) - dt.timedelta(days=1)
        t = c1 - L                                    # 1차 마감 회차
        plan.append((t, f'{n1} 마감', c1))
        while t - night_step >= night_start:
            t -= night_step
            plan.append((t, '밤', None))
        plan.append((max(c1, c2 - L), f'{n2} 마감', c2))   # 2차 마감 회차
        day_start = at(*DAY_START_KST)
        t = c2 + dt.timedelta(hours=1)                 # 장 마감 직후: 낮 시작까지 1시간 간격(7:30, 8:30)
        while t <= day_start:
            plan.append((t, '장 마감 직후', None))
            t += night_step
        t = day_start + day_step                       # 낮: 2시간 간격(10:30, 12:30, 14:30)
        while t < at(NIGHT_START_KST) and t + L <= at(NIGHT_START_KST) + dt.timedelta(minutes=30):
            plan.append((t, '낮', None))
            t += day_step
    plan = sorted(plan)
    plan = [(s, l, e if e is not None else (plan[i + 1][0] if i + 1 < len(plan) else s + L)) for i, (s, l, e) in enumerate(plan)]
    return [p for p in plan if p[0] > start]


def on_cutoff(label, nid, deadline):
    """마감: 받은 데까지로 다음 처리로 넘어간다. 표시 파일(cutoff_<이름>.json)을 남기고 알린다.
    news_process.ipynb의 대기 칸이 이 파일을 보고 그 보고 날짜를 처리한다."""
    d = night_dir(nid)
    items, hits, flog = load_night(nid)
    info = dict(label=label, report=nid, deadline_kst=deadline.isoformat(), done_kst=dt.datetime.now(KST).isoformat(),
                requests=len(flog), articles=int((items['drop'] == '').sum()))
    (d / f"cutoff_{label.split()[0]}.json").write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
    notify(f'[news_search] {label}({deadline:%H:%M}) → 받은 데까지로 처리 시작 표시: {summary_text(nid)}')


def run_scheduled(terms, until=None, log=print):
    """마감에 맞춘 일정으로 멈출 때까지 수집한다.

    - 회차 시작 시각은 plan_rounds로 잡는다. 회차 길이는 직전 회차의 실제 요청 속도로 다시 어림한다.
    - 검색 범위는 검색어마다 마지막으로 받은 때부터 지금까지 + 여유(term_window).
    - 회차마다 끝나야 할 시각이 있다(마감 회차는 마감, 나머지는 다음 회차 시작). 늦게 시작하면 남은 시간에 맞춰
      기사가 많은 검색어부터 하고, 그 시각이 되면 멈춘다(못 한 검색어는 다음 회차에서).
    - 다음 회차는 "방금 한 회차의 예정 시각" 뒤에서 고른다. 늦게 시작한 회차 때문에 다음 마감 회차가 사라지지 않고, 이미 지났으면 바로 시작한다.
    - 마감 회차가 끝나면(마감 시각이 되거나 다 마치면) on_cutoff로 마감 표시 파일을 남긴다. until(한국 시간)이 지나면 멈춘다.
    """
    client = GoogleNewsClient()
    n_terms = terms['term'].nunique()
    last_slot = None                                 # 방금 한 회차의 예정 시각
    notify(f'[news_search] 마감에 맞춘 수집 시작: 검색어 {n_terms}개, 1차 마감 {CUTOFFS[0][1]}:{CUTOFFS[0][2]:02d}, 2차 마감 {CUTOFFS[1][1]}:{CUTOFFS[1][2]:02d}')
    while until is None or dt.datetime.now(KST) < until:
        nid = report_id()
        _, _, flog = load_night(nid)
        round_min = n_terms * 3 * seconds_per_request(flog) / 60
        # 방금 한 회차의 예정 시각 10분 뒤부터의 첫 회차(이미 지났으면 바로 시작). 처음에는 지금부터.
        # 10분 여유: 회차 길이 어림이 바뀌어 같은 회차의 예정 시각이 몇 분 밀려도 두 번 하지 않게.
        plan = plan_rounds(last_slot + dt.timedelta(minutes=10) if last_slot else dt.datetime.now(KST) - dt.timedelta(minutes=1), round_min)
        start, label, end = plan[0]
        wait = (start - dt.datetime.now(KST)).total_seconds()
        if wait > 0:
            log(f'  다음 회차 {start:%m-%d %H:%M} ({label}, 회차 길이 약 {round_min:.0f}분) 까지 {wait / 60:.0f}분 기다림')
            time.sleep(min(wait, 600))               # 길어도 10분마다 일정을 다시 잡는다(속도가 바뀌면 반영)
            continue
        last_slot = start
        nid = report_id(start)                       # 보고 날짜는 예정 시각으로(늦게 시작한 2차도 그날 보고에)
        run_round(terms, client, nid=nid, log=log, todo=terms['term'].unique(), label=label, deadline=end)
        if '마감' in label:
            on_cutoff(label, nid, end)
    notify(f'[news_search] 수집 멈춤: {summary_text(report_id())}')


## 8. 검색어 표와 등급

검색어 표의 칸: `term`(검색 문자열에 넣을 검색어, 구문은 큰따옴표), `company`, `company_id`(KR:종목코드), `kind`(검색어 종류), `node`(그래프 노드: 해외 상대 `EN:`, 제품 `P:`, 브랜드 `B:`, 정책 `N:policy:`, 업종 `S:`, 테마 `T:`), `theme`(업종·테마 이름), `tier`(등급). 한 검색어가 여러 종목에 걸리면 종목마다 한 줄이고, 검색은 검색어마다 한 번만 한다.

- **지금은 임시로** 3차 시험 목록의 사본(`source/terms/test_queries.csv`, 시험 종목 18개)을 쓴다. 원본은 `news_search_tests/source/rss_test_short/`에 있고, 서비스가 시험 폴더를 읽지 않도록 2026-11-16에 복사해 두었다. 전체 검색어를 확정하면 실제 검색어 표로 바꾼다.
- **수집 종목**: `COLLECT_COMPANIES`(종목코드 목록)에 있는 종목의 검색어만 쓴다. 지금은 변수로 고르고, 나중에는 사용자 입력(파일)에서 읽는다. 수집·처리 노트북의 목록이 같아야 한다(다르면 처리 쪽이 모르는 검색어로 찾은 기사는 종목에 연결되지 않는다). 따옴표 없는 모양은 쓰지 않기로 해서 뺀다.
- **등급**: 정한 방식은 "가장 드문 등급에서 시작해 한 요청이 50건 이상이면 바로 한 단계 올리기"이고, 간격은 시험으로 정한다(`news_search.md` 2.3, 결정할 것 1). 지금 코드의 하루 기사 수 기준 등급(`tier_of`)은 그 전의 임시값이다.
- **2026-11-15 밤부터 시험**은 `DAY_NIGHT_TEST = True`로 등급 없이 모든 검색어를 시간대별 간격으로 검색한다. 검색어별 시간당 기사 수를 모아 등급 간격을 정하고, 요청 한도(밤 시간당 약 550건)도 함께 시험한다.

In [10]:
# ── 검색어 표: 3차 시험 목록 사본(임시, source/terms) + 하루 기사 수로 등급 ──
# 수집 종목(종목코드): 이 종목들의 검색어만 검색·처리한다. 지금은 변수로 고르고, 나중에는 사용자 입력(파일)에서 읽는다.
# 두 노트북(news_collect 8절, news_process 3절)이 같아야 한다. 지금 검색어 표(임시)에 있는 종목은 아래 18개뿐이다.
COLLECT_COMPANIES = [
    '000660',  # SK하이닉스
    '003490',  # 대한항공
    '004370',  # 농심
    '005490',  # POSCO홀딩스
    '009540',  # HD한국조선해양
    '012450',  # 한화에어로스페이스
    '042700',  # 한미반도체
    '068270',  # 셀트리온
    '087010',  # 펩트론
    '090430',  # 아모레퍼시픽
    '105560',  # KB금융
    '105840',  # 우진
    '114090',  # GKL
    '257720',  # 실리콘투
    '259960',  # 크래프톤
    '352820',  # 하이브
    '383800',  # LX홀딩스
    '425420',  # 티에프이
]
TQ = pd.read_csv(ROOT / 'source' / 'terms' / 'test_queries.csv', dtype=str).fillna('')
TQ = TQ[TQ['shape'] != '주제만·따옴표 없음']
# 검색어 = 검색 문자열에 넣는 모양(따옴표 포함). node·theme은 그래프 연결 근거(news_process.ipynb 6절)를 만들 때 쓴다
TERMS = TQ[['query', 'company', 'kind', 'node', 'theme']].rename(columns={'query': 'term'}).drop_duplicates(['term', 'company', 'kind'])
CO = pd.read_csv(PROJ / 'graph_db' / 'source' / 'graph' / 'neo4j' / 'nodes_Company.csv', dtype=str, usecols=['id', 'name']).fillna('')
TERMS['company_id'] = TERMS['company'].map(CO[CO['id'].str.startswith('KR:')].drop_duplicates('name').set_index('name')['id'])   # KR:종목코드
# 수집 종목만 남긴다
want = {'KR:' + c for c in COLLECT_COMPANIES}
no_terms = sorted(c for c in COLLECT_COMPANIES if 'KR:' + c not in set(TERMS['company_id']))
TERMS = TERMS[TERMS['company_id'].isin(want)].copy()
if no_terms:
    print(f'검색어 표에 없는 수집 종목 {len(no_terms)}개(검색하지 않음): {no_terms}')
n3 = pd.read_csv(ROOT / 'source' / 'terms' / 'queries.csv').set_index('query')['n']   # 3일 기사 수(본문 확보 매체)
TERMS['daily'] = TERMS['term'].map(n3).fillna(0) / 3
TERMS['tier'] = TERMS['daily'].map(tier_of)
DAY_NIGHT_TEST = True    # True: 등급 없이 모든 검색어를 시간대별(밤 1시간마다 최근 2시간, 낮 2시간마다 최근 3시간)로(2026-11-15 시험)
if DAY_NIGHT_TEST:
    TERMS['tier'] = '시간대별'
u = TERMS.drop_duplicates('term')
print(f"검색어 {len(u)}개(종목 {TERMS['company'].nunique()}개) / 등급별", u['tier'].value_counts().to_dict())
print(f'하루 예상 요청 {requests_per_day(TERMS):,}건(6초 간격이면 {requests_per_day(TERMS) * RSS_SLEEP / 3600:.1f}시간 분량, 24시간에 나눠서)')
display(u.sort_values('daily', ascending=False)[['term', 'tier', 'daily']].head(20))

검색어 183개(종목 18개) / 등급별 {'시간대별': 183}
하루 예상 요청 10,980건(6초 간격이면 18.3시간 분량, 24시간에 나눠서)


,term,tier,daily
5,"""Nvidia""",시간대별,401.666667
82,"""Microsoft""",시간대별,313.666667
140,"""natural gas""",시간대별,179.333333
158,"""Shell""",시간대별,106.000000
260,"""Treasury yields""",시간대별,71.333333
179,"""TSMC""",시간대별,68.000000
0,"""SK hynix""",시간대별,66.000000
37,"""Eli Lilly""",시간대별,59.666667
96,"""currency exchange""",시간대별,34.333333
142,"""Mercedes-Benz""",시간대별,33.333333


## 9. 한 회차 돌리기

차례가 된 검색어(이 보고 날짜 폴더에서 아직 안 받았거나, 마지막으로 받은 뒤 등급 간격이 지난 것)를 검색한다. 멈출 때까지 마감에 맞춰 이어 돌리려면 `run_scheduled(TERMS)`를 쓴다.

- `LIMIT`: 시험할 때 검색어 수를 줄인다(None이면 차례가 된 것 전부).
- 요청 수·시간을 확인하고 `RUN = True`로 바꿔 실행한다.

In [11]:
# ── 한 회차: 차례가 된 검색어 검색 ──
LIMIT = None
nid = report_id()
_, _, flog = load_night(nid)
todo = due_terms(TERMS, flog)
todo = todo[:LIMIT] if LIMIT else todo
print(f'보고 날짜 {nid}: 지금 차례인 검색어 {len(todo)}개 → 요청 {3 * len(todo)}건, 약 {3 * len(todo) * RSS_SLEEP / 60:.0f}분')
if RUN:
    run_round(TERMS, GoogleNewsClient(), nid=nid, limit=LIMIT)
else:
    print('RUN = False: 요청하지 않았다.')

보고 날짜 20261116: 지금 차례인 검색어 126개 → 요청 378건, 약 38분
RUN = False: 요청하지 않았다.


## 10. 쌓인 결과 보기

In [12]:
# ── 이 보고 날짜에 쌓인 것: 요청, 기사, 거른 것, 한도에 걸린 검색어 ──
items, hits, flog = load_night(nid)
if len(flog):
    print(f"요청 {len(flog)}건(성공 {(flog['status'].astype(str) == '200').sum()}), 회차 {flog['round'].nunique()}개, "
          f"기사 {len(items)}건(거른 것 {(items['drop'] != '').sum()}건: {items.loc[items['drop'] != '', 'drop'].value_counts().to_dict()})")
    print('매체별 기사', items[items['drop'] == '']['outlet'].value_counts().to_dict())
    cap = flog[flog['capped'].astype(bool)]
    if len(cap):
        print('100건 한도에 걸린 요청(등급을 올려야 한다):', cap[['term', 'group', 'window_h']].drop_duplicates().values.tolist())
else:
    print(f'보고 날짜 {nid}에 쌓인 것이 없다.')

요청 636건(성공 636), 회차 2개, 기사 148건(거른 것 5건: {'보도자료': 3, '스포츠 팬 사이트': 1, '시세·자료 페이지': 1})
매체별 기사 {'finance.yahoo.com': 64, 'theguardian.com': 38, 'fool.com': 18, 'cbsnews.com': 11, 'fortune.com': 6, 'cnn.com': 2, 'cnbc.com': 1, 'nypost.com': 1, 'businessinsider.com': 1, 'usatoday.com': 1}


## 11. 24시간 수집 돌리기

이 칸 하나로 서비스처럼 수집한다: 마감에 맞춘 일정(`run_scheduled`, 7절)으로 멈출 때까지 검색하고, 1·2차 마감 회차가 끝날 때마다 마감 표시 파일을 남긴다. 처리는 `news_process.ipynb`의 대기 칸을 함께 켜 두면 마감 표시를 보고 한다.

- 멈추려면 커널을 중지한다(검색어마다 바로 저장하므로 멈춘 데까지는 남는다).
- 외부 요청: 구글 뉴스 RSS(시험 종목 18개 기준 하루 약 11,000건, 밤 시간당 약 550건). `RUN_COLLECT = True`일 때만.
- 맥에서는 잠자기를 막아 둔다(전원 연결, 뚜껑 열기, `caffeinate`나 Amphetamine).

In [13]:
# ── 24시간 수집: 마감에 맞춘 일정으로 멈출 때까지 ──
RUN_COLLECT = False          # True여야 돌린다(멈출 때까지 계속)
if RUN_COLLECT:
    run_scheduled(TERMS)
else:
    print('RUN_COLLECT = False: 돌리지 않았다.')

RUN_COLLECT = False: 돌리지 않았다.
